In [1]:
# ==========================================
# CELL 1: TẢI TOÀN BỘ DỮ LIỆU & TỐI ƯU HÓA RAM
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
# Đã xóa import RandomForest và SMOTE vì không dùng nữa
import warnings
warnings.filterwarnings('ignore')

print("⏳ Đang tải TOÀN BỘ dữ liệu từ file CSV (Có thể mất 1-3 phút)...")
df_ml = pd.read_csv(r"D:\FPT\Spring 2026\DAP391m\DAP-Project\output\dashboard_data.csv")

df_sample = df_ml.copy()

print("🧹 Đang nén dữ liệu (Downcasting) để giải cứu 16GB RAM...")
# Kỹ thuật ép kiểu: Biến float64 thành float32, int64 thành int32 để giảm 50% dung lượng
for col in df_sample.columns:
    if df_sample[col].dtype == 'float64':
        df_sample[col] = df_sample[col].astype('float32')
    elif df_sample[col].dtype == 'int64':
        df_sample[col] = df_sample[col].astype('int32')

print(f"✅ Đã tải FULL dữ liệu thành công! Kích thước khổng lồ: {df_sample.shape}")

# Xóa bản gốc df_ml để giải phóng bộ nhớ RAM ngay lập tức
del df_ml

⏳ Đang tải TOÀN BỘ dữ liệu từ file CSV (Có thể mất 1-3 phút)...
🧹 Đang nén dữ liệu (Downcasting) để giải cứu 16GB RAM...
✅ Đã tải FULL dữ liệu thành công! Kích thước khổng lồ: (6985228, 20)


In [2]:
# ==========================================
# CELL 2: DATA CLEANING & ANTI-LEAKAGE
# ==========================================
print("🧹 Đang dọn dẹp các cột gây rò rỉ dữ liệu...")

# CHỈ XÓA 3 CỘT: Year (Thiên kiến lịch sử), Distance và Duration (Rò rỉ hậu quả)
cols_to_drop = ['Year', 'Distance(mi)', 'Duration']
df_sample = df_sample.drop(columns=[c for c in cols_to_drop if c in df_sample.columns], errors='ignore')

# Xóa các dòng khuyết thiếu dữ liệu để mô hình không bị lỗi toán học
df_sample = df_sample.dropna()

print(f"✅ Đã dọn dẹp xong! Kích thước dữ liệu sẵn sàng: {df_sample.shape}")
display(df_sample.head(3)) # In thử 3 dòng ra xem

🧹 Đang dọn dẹp các cột gây rò rỉ dữ liệu...
✅ Đã dọn dẹp xong! Kích thước dữ liệu sẵn sàng: (6985228, 17)


,Severity,Start_Lat,Start_Lng,City,State,Temperature(F),Humidity(%),Visibility(mi),Precipitation(in),Weather_Condition,Junction,Traffic_Signal,Sunrise_Sunset,Hour,Day,Month,Weekday
0,3,39.865147,-84.058723,Dayton,OH,36.900002,91.0,10.0,0.02,Light Rain,False,False,Night,5.0,8.0,2.0,0.0
1,2,39.928059,-82.831184,Reynoldsburg,OH,37.900002,100.0,10.0,0.00,Light Rain,False,False,Night,6.0,8.0,2.0,0.0
2,2,39.063148,-84.032608,Williamsburg,OH,36.000000,100.0,10.0,0.00,Overcast,False,True,Night,6.0,8.0,2.0,0.0


In [3]:
# ==========================================
# CELL 3: TRAIN/TEST SPLIT
# ==========================================
print("✂️ Đang chia tách dữ liệu Huấn luyện (Train) và Kiểm thử (Test)...")

X = df_sample.drop(columns=['Severity'])
y = df_sample['Severity']

# Chia tỷ lệ 80-20, dùng stratify để giữ nguyên tỷ lệ chênh lệch của các Mức độ tai nạn
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"📦 Tập Huấn luyện (Train) có: {X_train.shape[0]} dòng.")
print(f"📦 Tập Kiểm thử (Test) có: {X_test.shape[0]} dòng.")

✂️ Đang chia tách dữ liệu Huấn luyện (Train) và Kiểm thử (Test)...
📦 Tập Huấn luyện (Train) có: 5588182 dòng.
📦 Tập Kiểm thử (Test) có: 1397046 dòng.


In [5]:
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score

# QUAN TRỌNG: Import từ file transformers.py vừa tạo
from transformers import BinaryMapper, CyclicalEncoder, FrequencyEncoder

print("🛠️ Đang lắp ráp Pipeline...")

# 1. Định nghĩa cấu hình
bool_cols = ['Traffic_Signal', 'Junction']
time_cols = {'Hour': 24, 'Month': 12, 'Weekday': 7}
freq_cols = ['State', 'City', 'Weather_Condition']

# 2. Xây dựng Pipeline
ml_pipeline = Pipeline([
    ('binary_map', BinaryMapper(bool_cols=bool_cols)),
    ('cyclical_encode', CyclicalEncoder(time_cols=time_cols)),
    ('frequency_encode', FrequencyEncoder(freq_cols=freq_cols)),
    ('classifier', lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=63,
        class_weight='balanced',
        random_state=42
    ))
])

# 3. Huấn luyện
print("🧠 Đang huấn luyện mô hình...")
ml_pipeline.fit(X_train, y_train)

# 4. Xuất file model (File này sẽ mang theo "địa chỉ" của transformers.py)
joblib.dump(ml_pipeline, 'traffic_accident_pipeline.pkl')
print("✅ Đã lưu model: traffic_accident_pipeline.pkl")

# 5. Đánh giá nhanh
y_pred = ml_pipeline.predict(X_test)
print(f"🎯 Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

🛠️ Đang lắp ráp Pipeline...
🧠 Đang huấn luyện mô hình...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.089454 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1474
[LightGBM] [Info] Number of data points in the train set: 5588182, number of used features: 19
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
✅ Đã lưu model: traffic_accident_pipeline.pkl
🎯 Accuracy: 52.39%
